# Intelligent Expense Processing: A Guided Tutorial on Building Agents using AWS Strands with Llama4 Scout and Maverick on Amazon Bedrock 

## Overview
This comprehensive tutorial is designed to guide you through the process of developing and deploying  multi-agent systems using AWS Strands on Amazon Bedrock. Specifically, you will gain hands-on experience in:

1. Creating multiple autonomous agents leveraging the Llama4 Scout and Maverick models, integrated with specialized tools on Amazon Bedrock, to demonstrate the capabilities of AWS Strands.
2. Hosting a fully functional "Expense processing agent" utilizing the Amazon Bedrock AgentCore Runtime, showcasing the seamless integration of AI-driven automation and enterprise workflows.

By the end of this tutorial, you will have acquired the knowledge and skills necessary to design, develop, and deploy sophisticated agent-based applications on Amazon Bedrock with Llama4 models and AWS Strands.

### Tutorial Architecture

<center><img src='images/Agent_workshop.png'></center>

### Pre-requisites

The IAM role needed to run this notebook should contain the following permissions.

    a. AgentCore full access - Please refer to (https://docs.aws.amazon.com/aws-managed-policy/latest/reference/BedrockAgentCoreFullAccess.html)
    b. Amazon Bedrock - Please refer to (https://docs.aws.amazon.com/aws-managed-policy/latest/reference/AmazonBedrockFullAccess.html)
    c. IAM CreateRole, PutRole, CreatePolicy, and PutPolicy access.
    d. S3 CreateBucket (if not using SageMaker default bucket), GetObject, PutObject, and ListBucket access.

### Install Dependencies

In [2]:
# Install requirements
!pip install -r requirements.txt --force --no-cache --quiet

# Get SageMaker execution role (works in SageMaker) or use local AWS credentials
try:
    import sagemaker
    role = sagemaker.get_execution_role()
    print(f"SageMaker execution role: {role}")
except (ModuleNotFoundError, ValueError, AttributeError) as e:
    # Running locally - get role from AWS credentials or set manually
    import boto3
    sts = boto3.client('sts')
    try:
        # Try to get the caller identity to verify credentials
        identity = sts.get_caller_identity()
        account_id = identity.get('Account')
        arn = identity.get('Arn', '')
        
        # If running as a role, use that; otherwise you'll need to specify a role ARN
        if ':role/' in arn:
            role = arn
            print(f"Using IAM role from credentials: {role}")
        else:
            # Running as a user - you'll need to specify a role ARN manually
            # or the notebook will create one later for AgentCore Runtime
            print("Running locally. No SageMaker execution role available.")
            print("The notebook will create an IAM role for AgentCore Runtime later.")
            print(f"Current AWS identity: {arn}")
            role = None  # Will be set when creating AgentCore Runtime role
    except Exception as cred_error:
        print(f"Error getting AWS credentials: {cred_error}")
        print("Please configure AWS credentials (aws configure) or set environment variables.")
        role = None

role


[notice] A new release of pip is available: 23.0.1 -> 26.0
[notice] To update, run: pip install --upgrade pip
Running locally. No SageMaker execution role available.
The notebook will create an IAM role for AgentCore Runtime later.
Current AWS identity: arn:aws:iam::202658006492:user/chakde-dev


### Function to upload receipt images into S3 bucket

In [3]:
import boto3
import os

def upload_directory_to_s3(image_dir, bucket_name, s3_prefix):
    """
    Uploads all files from a local directory to an S3 bucket.

    Args:
        local_directory_path (str): The path to the local directory to upload.
        bucket_name (str): The name of the S3 bucket.
        s3_prefix (str, optional): An optional S3 key prefix to add to the uploaded files.
                                   This helps in organizing files within the S3 bucket.
    """
    s3_client = boto3.client('s3')

    if not os.path.isdir(image_dir):
        print(f"Error: Local directory '{image_dir}' not found.")
        return
    
    entries = os.listdir(image_dir)
    images = [image_dir+"/"+entry for entry in entries if os.path.isfile(os.path.join(image_dir, entry))]
    
    for img in images:
        try:
            s3_key = f"{s3_prefix}/{img.split('/')[-1]}"
            s3_client.upload_file(img, bucket_name, s3_key)
            print(f"Uploaded '{img}' to 's3://{bucket_name}/{s3_prefix}'")
        except Exception as e:
            print(f"Error uploading '{img}': {e}")



### Define S3 Bucket and Prefix where the images will be uploaded

In [4]:
# Replace with your actual directory path and S3 bucket details
images_dir = "./Receipt_images" #Local dir path where the receipt images are available

# Get S3 bucket - works in both SageMaker and local environments
try:
    import sagemaker
    session = sagemaker.Session()
    s3_bucket = session.default_bucket()
    print(f"Using SageMaker default bucket: {s3_bucket}")
except (ModuleNotFoundError, AttributeError, Exception) as e:
    # Running locally - use a bucket name or create default based on account
    import boto3
    sts = boto3.client('sts')
    try:
        account_id = sts.get_caller_identity().get('Account')
        region = boto3.session.Session().region_name or 'us-east-1'
        # Default bucket naming pattern (you can override this)
        s3_bucket = f"sagemaker-{region}-{account_id}"
        print(f"Running locally. Using default bucket name: {s3_bucket}")
        print("Note: Make sure this bucket exists or update s3_bucket with your bucket name.")
    except Exception as cred_error:
        print(f"Error getting AWS credentials: {cred_error}")
        # You can manually set the bucket name here
        s3_bucket = "your-bucket-name-here"  # Replace with your actual bucket name
        print(f"Using manually specified bucket: {s3_bucket}")

s3_prefix = "doc_processing/images" #Upload into a "directory" within the bucket


Running locally. Using default bucket name: sagemaker-us-east-1-202658006492
Note: Make sure this bucket exists or update s3_bucket with your bucket name.


In [5]:
#Upload all the images from local dir to S3
upload_directory_to_s3(images_dir, s3_bucket, s3_prefix)

Uploaded './Receipt_images/ticket_invoice.pdfpage_0.png' to 's3://sagemaker-us-east-1-202658006492/doc_processing/images'
Uploaded './Receipt_images/Taxi1.pdfpage_0.png' to 's3://sagemaker-us-east-1-202658006492/doc_processing/images'
Uploaded './Receipt_images/hotel.pdfpage_0.png' to 's3://sagemaker-us-east-1-202658006492/doc_processing/images'
Uploaded './Receipt_images/taxi2.pdfpage_0.png' to 's3://sagemaker-us-east-1-202658006492/doc_processing/images'
Uploaded './Receipt_images/meal2.pdfpage_0.png' to 's3://sagemaker-us-east-1-202658006492/doc_processing/images'
Uploaded './Receipt_images/meal1.pdfpage_0.png' to 's3://sagemaker-us-east-1-202658006492/doc_processing/images'


### Define employee related variables

In [6]:
import json
inputs = {"emp_first_name": "Jane",
           "emp_last_name": "Doe",
           "emp_num": 12345,
           "cost_center": 98763,
           "division": "Sales",
           "bucket": s3_bucket,
           "images_prefix": s3_prefix
        }

inputs = json.dumps(inputs)

#### Invoking local agent

In [8]:
from src.main_local import run_expense_processor
expense_report = run_expense_processor(inputs)


Tool #1: ocr
***** OCR - Started *****
***** OCR - Started *****
***** OCR - Completed *****

Tool #2: convert_currency_to_usd
***** Currency conversion - Started *****
***** Currency conversion - Completed *****

Tool #3: policy_compliant_check
***** Travel Compliance check - Started *****
***** Travel Compliance check - Completed *****
[{"type":"HOTEL","currency":"USD","guest_name":"Jane Doe","company":"XYZ Corporate","loyalty_number":"MS58943267","room_type":"Deluxe King","room_number":"1542","check_in":"05/14/2024","check_out":"05/16/2024","receipt_number":"86724531","charges":[{"date":"05/14/2024","description":"Room Charge - Deluxe King","amount":"$235.00"},{"date":"05/14/2024","description":"Room Service - Dinner","amount":"$85.50"},{"date":"05/15/2024","description":"Room Charge - Deluxe King","amount":"$235.00"},{"date":"05/15/2024","description":"Mini Bar","amount":"$42.75"},{"date":"05/15/2024","description":"In-Room Movie","amount":"$19.99"},{"date":"05/15/2024","descripti

In [9]:
#Function to read the output using pandas
import pandas as pd

def df_out(expense_report):
    df = pd.json_normalize(expense_report)
    return df

In [10]:
exp_report_sum = df_out(expense_report)
exp_report_sum

,Name,Employee_Number,Cost_Center,Division,Expenses,Status,TOTAL_DUE_TO_COMPANY,EXCEPTIONS_SUMMARY,COMPLIANT_STATUS
0,Jane Doe,12345,98763,Sales,"[{'type': 'HOTEL', 'currency': 'USD', 'guest_n...",Pending Approval,7799.71,[Nightly rate ($235.00) exceeds Tier 1 city li...,⚠️


In [11]:
#Exceptions details from DF
exp_report_sum['EXCEPTIONS_SUMMARY'][0]

['Nightly rate ($235.00) exceeds Tier 1 city limit ($220.00). Total incidentals ($186.74) exceed the $75 limit.',
 'Alcohol percentage (37.25%) exceeds the 20% limit. Total meal cost ($328.63) exceeds the daily meal allowance ($148).',
 'Total meal cost ($173.52) exceeds the daily meal allowance ($148).',
 'Potential non-compliance with flight class policy. Business Class is booked for a flight less than 8 hours. Employee level not specified.']

In [12]:
#Expenses details from DF 
exp_report_sum['Expenses'][0]

[{'type': 'HOTEL',
  'currency': 'USD',
  'guest_name': 'Jane Doe',
  'company': 'XYZ Corporate',
  'loyalty_number': 'MS58943267',
  'room_type': 'Deluxe King',
  'room_number': '1542',
  'check_in': '05/14/2024',
  'check_out': '05/16/2024',
  'receipt_number': '86724531',
  'charges': [{'date': '05/14/2024',
    'description': 'Room Charge - Deluxe King',
    'amount': '$235.00'},
   {'date': '05/14/2024',
    'description': 'Room Service - Dinner',
    'amount': '$85.50'},
   {'date': '05/15/2024',
    'description': 'Room Charge - Deluxe King',
    'amount': '$235.00'},
   {'date': '05/15/2024', 'description': 'Mini Bar', 'amount': '$42.75'},
   {'date': '05/15/2024', 'description': 'In-Room Movie', 'amount': '$19.99'},
   {'date': '05/15/2024',
    'description': 'Laundry Service',
    'amount': '$38.50'}],
  'subtotal': '$656.74',
  'tax': '$58.29',
  'total': 715.03,
  'TOTAL_USD': 715.03,
  'EXCEPTION': 'Nightly rate ($235.00) exceeds Tier 1 city limit ($220.00). Total inciden

### Preparing your agent for deployment on AgentCore Runtime

Let's now deploy our agents to AgentCore Runtime. To do so we need to add the following in out orchestrator agent ```src/main_AgentCore.py```:

Please check ```src/main_local.py```and  ```src/main_AgentCore.py``` on modifications done.

1. Import the Runtime App with ```from bedrock_agentcore.runtime import BedrockAgentCoreApp```
2. Initialize the App in our code with ```app = BedrockAgentCoreApp()```
3. Decorate the invocation function with the ```@app.entrypoint decorator```

Let AgentCoreRuntime control the running of the agent with ```app.run()```

#### Strands Agents with Amazon Bedrock model
Let's start with our Strands Agent using Amazon Bedrock model. All the others will work exactly the same.

In [13]:
#Local Version before making changes
!cat src/main_local.py 

import argparse
import ast
import json
import os
import pprint
import re
from typing import Any, Dict, List, Optional

import boto3
from pydantic import BaseModel, Field
from strands import Agent, tool
from strands.models import BedrockModel

from .cur_convert import convert_currency_to_usd
from .ocr_agent import ocr
from .policy_checker import policy_compliant_check
from .structured_out import run_final_sum

boto_session = boto3.session.Session()
region = boto_session.region_name
s3_resource = boto3.resource("s3")
bedrock_maverick_model = BedrockModel(
    model_id="us.meta.llama4-maverick-17b-instruct-v1:0",
    streaming=False,
    max_tokens=8192,
    temperature=0.0,
    top_p=0.3,
    region=region,
)

instruction = """
Task: Process invoices or receipts to extract and create expense reports.

Step-by-Step Instructions:
Extract Text: Use the 'ocr' tool to extract text from the provided image.
Convert Currency: Use the 'convert_currency_to_usd' tool to convert all currencies to US

In [14]:
!diff src/main_local.py src/main_AgentCore.py

1,3c1,10
< import argparse
< import ast
< import json
---
> from bedrock_agentcore.runtime import BedrockAgentCoreApp
> from strands import Agent, tool
> from strands.models import BedrockModel
> import boto3
> from ocr_agent import ocr
> from cur_convert import convert_currency_to_usd
> from policy_checker import policy_compliant_check
> from structured_out import run_final_sum
> from pydantic import BaseModel, Field
> from typing import List, Optional, Any, Dict
5c12
< import pprint
---
> import json
7c14,16
< from typing import Any, Dict, List, Optional
---
> import ast
> import argparse
> import pprint
9,12c18
< import boto3
< from pydantic import BaseModel, Field
< from strands import Agent, tool
< from strands.models import BedrockModel
---
> app = BedrockAgentCoreApp()
14,18d19
< from .cur_convert import convert_currency_to_usd
< from .ocr_agent import ocr
< from .policy_checker import policy_compliant_check
< from .structured_out import run_final_sum
< 
21c22
< s3_resource = bo

In [15]:
#After changes specific to AgentCore Runtime
!cat src/main_AgentCore.py 

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
import boto3
from ocr_agent import ocr
from cur_convert import convert_currency_to_usd
from policy_checker import policy_compliant_check
from structured_out import run_final_sum
from pydantic import BaseModel, Field
from typing import List, Optional, Any, Dict
import os
import json
import re
import ast
import argparse
import pprint

app = BedrockAgentCoreApp()

boto_session = boto3.session.Session()
region = boto_session.region_name
s3_resource = boto3.resource('s3')
bedrock_maverick_model = BedrockModel(
    model_id="us.meta.llama4-maverick-17b-instruct-v1:0",
    streaming=False,
    max_tokens=4096,
    temperature=0.,
    top_p=0.1,
    region=region
)

instruction = """
Task: Process invoices or receipts to extract and create expense reports.

Step-by-Step Instructions:
Extract Text: Use the 'ocr' tool to extract text from the provided image.
Convert C

#### Deploying the agent to AgentCore Runtime
The CreateAgentRuntime operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent.

Note: Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCore Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

#### Configure AgentCore Runtime deployment
First we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code

<center><img src='images/configure.png'></center>

#### IAM service role
To run an AgentCore Runtime you must create a service role and add all neccessary permissions. The service role allows AgentCore Runtime to perform actions on your behalf in your AWS account. Please refer to [Documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/security-iam.html) for more details.

In [17]:
#Create IAM role
iam = boto3.client('iam')
aws_acct = boto3.client('sts').get_caller_identity().get('Account')
region = boto3.session.Session().region_name

aws_acct
region


'us-east-1'

In [18]:
AC_policy_doc= json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AssumeRolePolicy",
            "Effect": "Allow",
            "Principal": {
                "Service": "bedrock-agentcore.amazonaws.com"
            },
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {
                    "aws:SourceAccount": "{}".format(aws_acct)
                },
                "ArnLike": {
                    "aws:SourceArn": "arn:aws:bedrock-agentcore:{}:{}:*".format(region, aws_acct)
                }
            }
        }
    ]
}
)

In [19]:
import datetime

role_name="AgentCoreRuntimeWorkshop-{}".format(str(datetime.datetime.now().timestamp()).split('.')[0])
create_role_response = iam.create_role(
    RoleName=role_name,
    AssumeRolePolicyDocument = AC_policy_doc
)

ClientError: An error occurred (InvalidClientTokenId) when calling the CreateRole operation: The security token included in the request is invalid

In [ ]:
role_arn = create_role_response["Role"]["Arn"]

role_arn

In [ ]:
AC_permission_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "ECRImageAccess",
            "Effect": "Allow",
            "Action": [
                "ecr:BatchGetImage",
                "ecr:GetDownloadUrlForLayer"
            ],
            "Resource": [
                "arn:aws:ecr:{}:{}:repository/*".format(region, aws_acct)
            ]        
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:DescribeLogStreams",
                "logs:CreateLogGroup"
            ],
            "Resource": [
                "arn:aws:logs:{}:{}:log-group:/aws/bedrock-agentcore/runtimes/*".format(region, aws_acct)
            ]
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:DescribeLogGroups"
            ],
            "Resource": [
                "arn:aws:logs:{}:{}:log-group:*".format(region, aws_acct)
            ]
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogStream",
                "logs:PutLogEvents"
            ],
            "Resource": [
                "arn:aws:logs:{}:{}:log-group:/aws/bedrock-agentcore/runtimes/*:log-stream:*".format(region, aws_acct)
            ]
        },
        {
            "Sid": "ECRTokenAccess",
            "Effect": "Allow",
            "Action": [
                "ecr:GetAuthorizationToken"
            ],
            "Resource": "*"
        },
        {
        "Effect": "Allow", 
        "Action": [ 
            "xray:PutTraceSegments", 
            "xray:PutTelemetryRecords", 
            "xray:GetSamplingRules", 
            "xray:GetSamplingTargets"
            ],
         "Resource": [ "*" ] 
         },
         {
            "Effect": "Allow",
            "Resource": "*",
            "Action": "cloudwatch:PutMetricData",
            "Condition": {
                "StringEquals": {
                    "cloudwatch:namespace": "bedrock-agentcore"
                }
            }
        },
        {
            "Sid": "GetAgentAccessToken",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:GetWorkloadAccessToken",
                "bedrock-agentcore:GetWorkloadAccessTokenForJWT",
                "bedrock-agentcore:GetWorkloadAccessTokenForUserId"
            ],
            "Resource": [
              "arn:aws:bedrock-agentcore:{}:{}:workload-identity-directory/default".format(region, aws_acct),
              "arn:aws:bedrock-agentcore:{}:{}:workload-identity-directory/default/workload-identity/agentName-*".format(region, aws_acct)
            ]
        },
         {
			"Sid": "BedrockModelInvocation",
			"Effect": "Allow",
			"Action": [
				"bedrock:InvokeModel",
				"bedrock:InvokeModelWithResponseStream",
				"bedrock:ApplyGuardrail"
			],
			"Resource": [
				"arn:aws:bedrock:*::foundation-model/*",
				"arn:aws:bedrock:{}:{}:*".format(region, aws_acct)
			]
		},
        {
			"Sid": "ListObjectsInBucket",
			"Effect": "Allow",
			"Action": [
				"s3:ListBucket"
			],
			"Resource": [
				"arn:aws:s3:::{}".format(s3_bucket)
			]
		},
		{
			"Sid": "AllObjectActions",
			"Effect": "Allow",
			"Action": "s3:*Object",
			"Resource": [
				"arn:aws:s3:::{}/*".format(s3_bucket)
			]
		}
    ]
}

                          )


In [ ]:
iam_ac_response = iam.put_role_policy(
    RoleName=role_name,
    PolicyName="agentcore-perm",
    PolicyDocument=AC_permission_policy
)
iam_ac_response

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import datetime
import random

boto_session = Session()
region = boto_session.region_name

# Get the current datetime
now = datetime.datetime.now()

# Format the datetime into a string (e.g., 'YYYYMMDD_HHMMSS')
datetime_string = now.strftime("%Y%m%d%H%M%S")

# Generate a random integer (e.g., between 1000 and 9999)
random_number = random.randint(1000, 9999)

# Combine the datetime string and the random number into a single string
random_string = f"{datetime_string}_{random_number}"

agentcore_runtime = Runtime()
agent_name = f"expense_claim_processor_{random_string}"
response = agentcore_runtime.configure(
    entrypoint="src/main_AgentCore.py",
    execution_role=role_arn,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name
)


#### Launching agent to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime.

<center><img src='images/launch_1.png'></center>

#### Write a custom Dockerfile

In [ ]:
%%writefile Dockerfile
FROM --platform=linux/arm64 ghcr.io/astral-sh/uv:python3.12-bookworm-slim
WORKDIR /app

#COPY all dependencies
COPY src/main_AgentCore.py /app/main_AgentCore.py
COPY src/ocr_agent.py /app/ocr_agent.py
COPY src/policy_checker.py /app/policy_checker.py
COPY src/cur_convert.py /app/cur_convert.py
COPY src/structured_out.py /app/structured_out.py
COPY src/travel_policy.txt /app/travel_policy.txt

# Configure UV for container environment
ENV UV_SYSTEM_PYTHON=1 UV_COMPILE_BYTECODE=1

COPY requirements.txt requirements.txt
# Install from requirements file
RUN uv pip install -r requirements.txt
RUN uv pip install aws-opentelemetry-distro>=0.10.1

# Set AWS region environment variable
ENV AWS_REGION=us-east-1
ENV AWS_DEFAULT_REGION=us-east-1

# Signal that this is running in Docker for host binding logic
ENV DOCKER_CONTAINER=1

# Create non-root user
RUN useradd -m -u 1000 bedrock_agentcore
USER bedrock_agentcore

EXPOSE 8080
EXPOSE 8000

# Copy entire project (respecting .dockerignore)
COPY . .

# Use the full module path
CMD ["opentelemetry-instrument", "python", "-m", "main_AgentCore"]

#### Launching agent to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime
<center><img src='images/launch_2.png'></center>

In [ ]:
launch_result = agentcore_runtime.launch()

#### Checking for the AgentCore Runtime Status
Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

#### Invoking AgentCore Runtime
Finally, we can invoke our AgentCore Runtime with a payload
<center><img src='images/invoke.png'></center>

In [ ]:
import json
payload = {"emp_first_name": "Jane",
           "emp_last_name": "Doe",
           "emp_num": 12345,
           "cost_center": 98763,
           "division": "Sales",
           "bucket": s3_bucket,
           "images_prefix": s3_prefix
        }

payload = json.dumps(payload)

In [ ]:
invoke_response = agentcore_runtime.invoke(payload)


#### Processing invocation results
We can now process our invocation results to include it in an application

In [ ]:
response_json = json.loads("".join(invoke_response['response']))
df_result = df_out(response_json)
df_result

#Note: You can also explore the expenses within results as df_result['Expenses'] and df_result['EXCEPTIONS_SUMMARY']

#### Invoking AgentCore Runtime with boto3
Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 ```invoke_agent_runtime``` method for it.

In [ ]:
import boto3
agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=payload
)
if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]


In [ ]:
boto3_response_json = json.loads("".join([event.decode("utf-8") for event in events]))
df_result = df_out(boto3_response_json)
df_result

#Note: You can also explore the expenses within results as df_result['Expenses'] and df_result['EXCEPTIONS_SUMMARY']

### Cleanup (Optional)
Let's now clean up the AgentCore Runtime created

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

In [ ]:
def cleanup_all_resources():
    """Delete all resources created during the notebook execution"""
    
    # Initialize clients
    agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
    ecr_client = boto3.client('ecr', region_name=region)
    s3_client = boto3.client('s3')
    iam_client = boto3.client('iam')
    codebuild_client = boto3.client('codebuild', region_name=region)
    
    # 1. Delete AgentCore Runtime
    try:
        print("Deleting AgentCore Runtime...")
        agentcore_control_client.delete_agent_runtime(agentRuntimeId=launch_result.agent_id)
        print(f" AgentCore Runtime {launch_result.agent_id} deleted")
    except Exception as e:
        print(f" AgentCore Runtime: {str(e)}")
    
    # 2. Delete ECR Repository
    try:
        print("Deleting ECR Repository...")
        ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split('/')[1], force=True)
        print(f" ECR Repository deleted")
    except Exception as e:
        print(f" ECR Repository: {str(e)}")
    
    # 3. Delete CodeBuild Project
    try:
        print("Deleting CodeBuild Project...")
        projects = codebuild_client.list_projects()['projects']
        agent_project = next((p for p in projects if launch_result.agent_id.split('_')[0] in p), None)
        if agent_project:
            codebuild_client.delete_project(name=agent_project)
            print(f" CodeBuild Project {agent_project} deleted")
        else:
            print("No CodeBuild project found")
    except Exception as e:
        print(f"CodeBuild Project: {str(e)}")
    
    # 4. Delete CodeBuild IAM Role (with policies)
    try:
        print("Deleting CodeBuild IAM Role...")
        roles = iam_client.list_roles()['Roles']
        codebuild_role = next((r['RoleName'] for r in roles if 'AmazonBedrockAgentCoreSDKCodeBuild' in r['RoleName']), None)
        if codebuild_role:
            # Delete attached policies
            policies = iam_client.list_attached_role_policies(RoleName=codebuild_role)
            for policy in policies['AttachedPolicies']:
                iam_client.detach_role_policy(RoleName=codebuild_role, PolicyArn=policy['PolicyArn'])
            # Delete inline policies
            inline_policies = iam_client.list_role_policies(RoleName=codebuild_role)
            for policy_name in inline_policies['PolicyNames']:
                iam_client.delete_role_policy(RoleName=codebuild_role, PolicyName=policy_name)
            # Delete role
            iam_client.delete_role(RoleName=codebuild_role)
            print(f"CodeBuild IAM Role {codebuild_role} deleted")
        else:
            print("No CodeBuild IAM role found")
    except Exception as e:
        print(f"CodeBuild IAM Role: {str(e)}")
    
    # 5. Delete CodeBuild S3 Bucket (empty first)
    try:
        print("Deleting CodeBuild S3 Bucket...")
        buckets = s3_client.list_buckets()['Buckets']
        codebuild_bucket = next((b['Name'] for b in buckets if 'bedrock-agentcore-codebuild-sources' in b['Name']), None)
        if codebuild_bucket:
            # Empty bucket
            objects = s3_client.list_objects_v2(Bucket=codebuild_bucket)
            if 'Contents' in objects:
                delete_keys = [{'Key': obj['Key']} for obj in objects['Contents']]
                s3_client.delete_objects(Bucket=codebuild_bucket, Delete={'Objects': delete_keys})
            # Delete bucket
            s3_client.delete_bucket(Bucket=codebuild_bucket)
            print(f"CodeBuild S3 Bucket {codebuild_bucket} deleted")
        else:
            print("No CodeBuild S3 bucket found")
    except Exception as e:
        print(f"CodeBuild S3 Bucket: {str(e)}")
    
    # 6. Delete uploaded S3 objects
    try:
        print("Deleting uploaded S3 objects...")
        objects = s3_client.list_objects_v2(Bucket=s3_bucket, Prefix=s3_prefix)
        if 'Contents' in objects:
            delete_keys = [{'Key': obj['Key']} for obj in objects['Contents']]
            s3_client.delete_objects(Bucket=s3_bucket, Delete={'Objects': delete_keys})
            print(f"Deleted {len(delete_keys)} S3 objects")
        else:
            print("No S3 objects found to delete")
    except Exception as e:
        print(f"S3 cleanup: {str(e)}")
    
    # 7. Delete main IAM Role and Policy
    try:
        print("Deleting main IAM Role and Policy...")
        iam_client.delete_role_policy(RoleName=role_name, PolicyName="agentcore-perm")
        iam_client.delete_role(RoleName=role_name)
        print(f"IAM Role {role_name} deleted")
    except Exception as e:
        print(f"IAM cleanup: {str(e)}")
    
    print("\nComplete cleanup finished!")

# Run complete cleanup
cleanup_all_resources()
